# Mesure de transmission de la lentille

https://rubinobs.atlassian.net/wiki/spaces/LTS/pages/50086854/CBP+Calibration+System+Hardware#Lens


In [ ]:
import os 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import pickle
from astropy.constants import e, h, c
from scipy.interpolate import interp1d
import pandas as pd
from tcbp.reduction.dataset import get_calib_path

from tcbp.analysis.lsst import LSSTRun

%matplotlib widget

In [ ]:
def get_data(run, doplot=True):
    """
    Compute and plot charges ratio between sphere and receiver photodiodes for one run

    run : LSSTRun
        LSSTRun object containing the data for the run
    doplot : bool
        If True, the results are plotted
    """

    ### Get data
    wls = run["set_wl"]
    
    Q_sphere = run['pd_charge_per_burst'][wls > 0.]
    Q_receiver = run['calib_charge_per_burst'][wls > 0.]

    Q_sphere_err = run['pd_charge_per_burst_err'][wls > 0.]
    Q_receiver_err = run['calib_charge_per_burst_err'][wls > 0.]

    ratio = np.abs(Q_receiver / Q_sphere)
    ratio_err = np.abs(ratio) * np.sqrt((Q_sphere_err / Q_sphere)**2 + (Q_receiver_err / Q_receiver)**2)

    ### Plot
    if doplot:
        fig, (ax1,ax2,ax3) = plt.subplots(3, sharex="all", figsize=(8,8))

        ax1.errorbar(wls[wls > 0.], Q_sphere, yerr=Q_sphere_err, c='b', marker='.', ls="")
        ax1.set_ylabel('Charges Sphere [C]')
            
        ax2.errorbar(wls[wls > 0.], Q_receiver, yerr=Q_receiver_err, c='r', marker='.', ls="")
        ax2.set_ylabel('Charges Receiver [C]')

        ax3.errorbar(wls[wls > 0.], ratio, yerr=ratio_err, c='indigo', marker='.', ls="")
        ax3.set_ylabel('Ratio Receiver / Sphere [No unit]')
        ax3.set_xlabel('wavelengths [nm]')

        fig.tight_layout()
        plt.show()

    return ratio, ratio_err

## Data file

In [ ]:
main_repo = '/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/data/'

### Avec la cellule solaire
#data_dir = main_repo + '20260701_optical_bench_lens_transmission/'  # Tout premier run, très bruité à cause des long temps de pause
#data_dir = main_repo + '20260701_171854_optical_bench_lens_transmission/' # Avec lentille, scan complet, petits tps de pause x 5
#data_dir = main_repo + '20260701_184358_optical_bench_lens_transmission/' # Avec lentille, de 300 à 400nm seulement, petits tps de pause x 5
#data_dir = main_repo + '20260701_193647_optical_bench_lens_transmission/' # Sans lentille, scan complet, petits tps de pause x 5

### Avec la mini lentille
#data_dir = main_repo + '20260703_145059_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/' # Sans grosse lentille
#data_dir = main_repo + '20260703_163504_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260703_190329_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/' # Avec grosse lentille  Le montage a bougé entre les deux runs, donc pas comparable
#data_dir = main_repo + '20260703_200311_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/' # Sans grosse lentille
#data_dir = main_repo + '20260706_140844_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260706_185224_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/' # Sans grosse lentille
#data_dir = main_repo + '20260706_195018_optical_bench_lens_transmission_pinhole5mm_MiniLensBigLensReversed_slit20/' # Avec grosse lentille retournée
#data_dir = main_repo + '20260707_174705_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensWithBigOpenIris_slit20/' # Avec grosse lentille + iris ouvert 
#data_dir = main_repo + '20260707_193623_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensWithBigCloseIris_slit20/' # Avec grosse lentille + iris fermé
#data_dir = main_repo + '20260707_203859_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensNoBigCloseIris_slit20/' # Sans grosse lentille + iris fermé
data_dir = main_repo + '20260707_213549_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensNoBigOpenIris_slit20/' # Sans grosse lentille + iris ouvert

### En projetant l'image de la mini lens par la grosse lentille
#data_dir = main_repo + '20260706_154719_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBigLongProj_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260706_170942_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBigLongProjMask_slit20/' # Avec grosse lentille + mask


### Image d'un pinhole de 1mm sur photodiode (pas de mini lens) => Très bruité
#data_dir = main_repo + '20260706_215451_optical_bench_lens_transmission_pinhole1mm_BigLensLongProj_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260706_234151_optical_bench_lens_transmission_pinhole1mm_NoLensShortProj_slit20/' # Sans grosse lentille, photodiode à la sortie du tube


### Avec le doublet achromatique
#data_dir = main_repo + '20260707_134736_optical_bench_lens_transmission_pinhole1mm_DoubletAcNoBigLens_slit20/' # Avec doublet achromatique, sans grosse lentille
#data_dir = main_repo + '20260707_152515_optical_bench_lens_transmission_pinhole1mm_DoubletAcWithBigLens_slit20/' # Avec doublet achromatique, avec grosse lentille

## Reduce the data

Here we build the run.py file.

In [ ]:
pd_extname="KEITHLEY" # Sphere
sc_extname="KEYSIGHT" # NIST

## Do the reduction
run = LSSTRun(data_dir,
              nbursts=1, spectro_calib_path="2026_Auxtel/20260429", 
              spectro_temp_calib=False,
              pd_extname=pd_extname, sc_extname=sc_extname,
              pins={"KEYSIGHT":1, "KEITHLEY":4, "SHUTTER":2},
              ticks_per_shutter_event=1,
              use_digital_analyzer=True)
run.load()
run.extraction()
run.plot_summary()

np.save(data_dir + "run.npy", run.data)

In [ ]:
### Load a run file
run = np.load(data_dir + "run.npy", allow_pickle=True)
wls_run = run["set_wl"][run["set_wl"] > 300.]
ratio, ratio_err = get_data(run, doplot=True)

print(np.mean(np.abs(ratio_err) / ratio))

## Lens transmission measured by Elana

In [ ]:
data_dir2 = '/home/lmousset/Projets_Recherche/LSST/Mesure_transmission_tcbp/'


csv_file = data_dir2 + 'lens_ratio_calibration_system_2.csv'

df_lens = pd.read_csv(csv_file)
print(df_lens.head())

wls_lens = df_lens['Wavelength'].values
transmission_lens = df_lens['lens_ratio'].values
lens_ratio_error = df_lens['ratio_error'].values

plt.figure(figsize=(10,5))
plt.plot(wls_lens, transmission_lens, c='orange', marker='.', ls="")
plt.ylim(0, 1)


## Get several runs

In [ ]:
data_dict = {'20260701_171854': ['Solar cell', 'Avec lentille'],
             '20260701_184358': ['Solar cell', 'Avec lentille, de 300 à 400nm seulement'],
             '20260701_193647': ['Solar cell', 'Sans lentille'],

             '20260703_145059': ['Mini lens', 'Sans grosse lentille'],
             '20260703_163504': ['Mini lens', 'Avec grosse lentille'],

             '20260703_190329': ['Mini lens', 'Avec grosse lentille  Le montage a bougé entre les deux runs, donc pas comparable'],
             '20260703_200311': ['Mini lens', 'Sans grosse lentille'],
             '20260706_140844': ['Mini lens', 'Avec grosse lentille'],

             '20260706_154719': ['Mini lens', 'Avec grosse lentille'],
             '20260706_170942': ['Mini lens', 'Avec grosse lentille + mask'],
             '20260706_185224': ['Mini lens', 'Sans grosse lentille'],
             '20260706_195018': ['Mini lens', 'Avec grosse lentille retournée'],
             '20260706_215451': ['Mini lens', 'Avec grosse lentille'],
             '20260706_234151': ['Mini lens', 'Sans grosse lentille, photodiode à la sortie du tube'],
             '20260707_134736': ['Doublet achromatique', 'Avec doublet achromatique, sans grosse lentille'],
             '20260707_152515': ['Doublet achromatique', 'Avec doublet achromatique, avec grosse lentille'],
             '20260707_174705': ['Mini lens', 'Avec grosse lentille + iris ouvert'],
             '20260707_193623': ['Mini lens', 'Avec grosse lentille + iris fermé'],
             '20260707_203859': ['Mini lens', 'Sans grosse lentille + iris fermé'],
             '20260707_213549': ['Mini lens', 'Sans grosse lentille + iris ouvert']
             }

### Association des runs pour le calcul de la transmission de la lentille
### key = avec grosse lentille, value = [sans grosse lentille, description]
ratio_dict = {'20260701_171854': ['20260701_193647', 'Solar cell'],
              '20260701_184358': ['20260701_193647', 'Solar cell de 300 à 400nm'],
              '20260703_163504': ['20260703_145059', 'Mini lens'],
              '20260706_140844': ['20260703_200311', 'Mini lens'],
              '20260706_195018': ['20260706_185224', 'Mini lens - Big lens reversed'],
              '20260707_174705': ['20260707_213549', 'Mini lens - Iris ouvert'],
              '20260707_193623': ['20260707_203859', 'Mini lens - Iris fermé'],
              '20260707_152515': ['20260707_134736', 'Doublet AC']
}


In [ ]:
nruns = len(data_dict.keys())

for i, key in enumerate(data_dict.keys()):
    print(f'Run {i}: {key}')
    matching_dirs = [d for d in os.listdir(main_repo) if d.startswith(key)]

    data_dir = os.path.join(main_repo, matching_dirs[0])
    print(data_dir)

    run = np.load(data_dir + "/run.npy", allow_pickle=True)

    wls_run = run["set_wl"][run["set_wl"] > 0.]

    #print(wls_run)
    ratio, ratio_err = get_data(run, doplot=False)

    ### Reorder the arrays by wavelength and append to the data_dict
    sort_indices = np.argsort(wls_run)
    data_dict[key].append(wls_run[sort_indices])
    data_dict[key].append(ratio[sort_indices])
    data_dict[key].append(ratio_err[sort_indices])


In [ ]:
## Plot all ratios
plt.figure(figsize=(10,5))
colors = plt.cm.tab20(np.linspace(0, 1, len(data_dict)))
for i, (key, color) in enumerate(zip(data_dict.keys(), colors)):
    plt.errorbar(data_dict[key][2], data_dict[key][3], yerr=np.abs(data_dict[key][4]), marker='.', ls="", color=color, label=f'{data_dict[key][0]}: {data_dict[key][1]}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
plt.legend(fontsize=8)

In [ ]:
for i, (key, color) in enumerate(zip(data_dict.keys(), colors)):
    print("Run", i)
    if len(data_dict[key][2]) == len(np.unique(data_dict[key][2])):
        print("Wavelengths are unique, no need to average")
        continue
    else:
        wls_set = np.unique(data_dict[key][2])
        avg = [np.mean(data_dict[key][3][data_dict[key][2]==k]) for k in wls_set]
        err_avg = [np.std(data_dict[key][4][data_dict[key][2]==k])/np.sqrt(len(data_dict[key][4][data_dict[key][2]==k])) for k in wls_set]
        # On remplace les valeurs dans le dictionnaire par les valeurs moyennes
        data_dict[key][2] = wls_set
        data_dict[key][3] = np.array(avg)
        data_dict[key][4] = np.array(err_avg)

In [ ]:
## Plot all ratios
plt.figure(figsize=(10,5))
colors = plt.cm.tab20(np.linspace(0, 1, nruns))
for i, (key, color) in enumerate(zip(data_dict.keys(), colors)):
    print(i)    
    
    plt.errorbar(data_dict[key][2], data_dict[key][3], yerr=np.abs(data_dict[key][4]), marker='.', ls="", color=color, label=f'{data_dict[key][0]}: {data_dict[key][1]}')

plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
plt.legend(fontsize=8)

## Lens Transmission

In [ ]:
## Rapport entre 2 scans
plt.figure(figsize=(10,5))

# Loop sur ceux avec la grosse lentille
for i, key1 in enumerate(ratio_dict.keys()):
    key2 = ratio_dict[key1][0] # Run sans grosse lentille
    print(f'Run {i}: {key1} / {key2}')
    wls1 = data_dict[key1][2]
    wls2 = data_dict[key2][2]
    ratio1 = data_dict[key1][3]
    ratio2 = data_dict[key2][3]
    ratio_err1 = data_dict[key1][4]
    ratio_err2 = data_dict[key2][4]
    

    commun_wls = np.intersect1d(wls1, wls2)
    print(f"Commun wavelengths between run {key1} and run {key2}: {commun_wls}")

    # Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
    indices1 = np.searchsorted(wls1, commun_wls)
    indices2 = np.searchsorted(wls2, commun_wls)

    commun_ratio1 = ratio1[indices1]
    commun_ratio2 = ratio2[indices2]

    commun_ratio_err1 = ratio_err1[indices1]
    commun_ratio_err2 = ratio_err2[indices2]

    rapport = commun_ratio1 / commun_ratio2
    rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport
    print(rapport_err)

    ratio_dict[key1].append(commun_wls)
    ratio_dict[key1].append(rapport)
    ratio_dict[key1].append(rapport_err)

    plt.errorbar(commun_wls, rapport, yerr=rapport_err, marker='.', ls="", label=f'{ratio_dict[key1][1]}')

plt.errorbar(wls_lens, transmission_lens, yerr=lens_ratio_error, c='k', marker='.', ls="--",  label='Elana measurement')
plt.ylim(0., 1)
plt.ylabel('Lens transmission')
plt.xlabel('wavelengths [nm]')
plt.legend()


   

In [ ]:
# Save the ratio_dict in a pickle file
with open("/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/lens_transmission_ratio_dict.pkl", "wb") as f:
    pickle.dump(ratio_dict, f)